# Module 08 — Adversarial Patch Generator
## Attacks on Multimodal LLMs: CLIP Gradient-Based Adversarial Patches

---

This notebook demonstrates the mathematical foundation of adversarial attacks on vision-language models.
You will:

1. Understand how CLIP maps images and text into a shared embedding space
2. Craft an adversarial patch via gradient ascent on the CLIP image-text similarity
3. Visualise how the patch shifts an image's embedding toward a target text description
4. Export the patch for use in Challenge 4 (Phantom Patch)
5. Survey defences: feature squeezing, adversarial training, randomised smoothing

**Requirements:** `torch torchvision transformers Pillow matplotlib`

```bash
pip install torch torchvision transformers Pillow matplotlib
```

> **Hardware:** Runs on CPU. Apple Silicon (MPS) or CUDA GPU gives ~5–10× speedup.


In [ ]:
import torch
import torchvision.transforms as T
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from transformers import CLIPProcessor, CLIPModel
import io, os

# ── Device ──────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Using device: {DEVICE}')

---

## 1. How Multimodal LLMs Process Images

Vision-Language Models (VLMs) like LLaVA use a two-stage architecture:

```
Image  ──► Vision Encoder (CLIP ViT) ──► Image Embeddings ──┐
                                                              ├──► LLM ──► Text
Text   ──► Tokeniser ─────────────────► Token Embeddings  ──┘
```

**CLIP (Contrastive Language-Image Pre-training)** is the key component:
- Trained on 400M image-text pairs to align visual and linguistic representations
- Maps images and text to a shared **embedding space** (ℝ^512 for CLIP-ViT-B/32)
- Similar concepts have **high cosine similarity** regardless of modality

**Why this matters for adversarial attacks:**
- The LLM's understanding of image content comes entirely from CLIP embeddings
- If we can shift an image's embedding toward a different text description, the LLM "sees" something different
- We do this via **gradient ascent** on the cosine similarity between the image embedding and a target text embedding


In [ ]:
# ── Load CLIP ────────────────────────────────────────────────────────────────
# openai/clip-vit-base-patch32: 87M parameters, CPU-friendly, ~580 MB download

print('Loading CLIP (openai/clip-vit-base-patch32)...')
model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE)
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
model.eval()

print('CLIP loaded.')
print(f'  Embedding dimension: {model.config.projection_dim}')

---

## 2. The CLIP Embedding Space

Let's explore how CLIP maps images and text to the same space,
and visualise the cosine similarity between example concepts.


In [ ]:
# ── Helper: get normalised text embedding ────────────────────────────────────

def get_text_embedding(texts):
    """Return L2-normalised text embeddings from CLIP, shape (N, D)."""
    inputs = processor(text=texts, return_tensors='pt', padding=True).to(DEVICE)
    with torch.no_grad():
        feats = model.get_text_features(**inputs)
    return torch.nn.functional.normalize(feats, dim=-1)


def get_image_embedding(img_pil):
    """Return L2-normalised image embedding from CLIP, shape (1, D)."""
    inputs = processor(images=img_pil, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        feats = model.get_image_features(**inputs)
    return torch.nn.functional.normalize(feats, dim=-1)


# ── Visualise text-text similarity ──────────────────────────────────────────

labels = [
    'a violent image with weapons',
    'a dangerous explosion',
    'an image showing harm',
    'a harmless technical diagram',
    'a benign calibration pattern',
    'a scientific measurement chart',
]

embeds = get_text_embedding(labels)  # (N, D)
sim = (embeds @ embeds.T).cpu().numpy()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim, cmap='RdYlGn', vmin=-0.2, vmax=1.0)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels([l[:30] for l in labels], rotation=45, ha='right', fontsize=8)
ax.set_yticklabels([l[:30] for l in labels], fontsize=8)
ax.set_title('CLIP text-text cosine similarity\n(green = similar, red = dissimilar)', fontsize=11)
plt.colorbar(im, ax=ax)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=7,
                color='black' if sim[i,j] < 0.7 else 'white')
plt.tight_layout()
plt.show()

print('\nKey insight:')
print(f'  "violent" ↔ "dangerous":     {sim[0,1]:.3f}  (high — same cluster)')
print(f'  "violent" ↔ "technical":     {sim[0,3]:.3f}  (low  — different clusters)')
print(f'  "technical" ↔ "calibration": {sim[3,4]:.3f}  (high — same cluster)')

---

## 3. Adversarial Patch Optimisation

### The attack objective

We want to craft a small patch $\delta$ (a portion of the image) such that:

$$\max_{\delta} \; \cos\!\left(f_{\!I}(x + M \odot \delta),\; f_{\!T}(t_{\text{target}})\right)$$

where:
- $f_{\!I}$ = CLIP image encoder
- $f_{\!T}$ = CLIP text encoder  
- $x$ = original image
- $M$ = binary mask (where the patch is applied)
- $\delta$ = the learnable patch (pixel values, constrained to [0, 1])
- $t_{\text{target}}$ = target text description (e.g. "a harmless technical diagram")

We optimise $\delta$ with gradient ascent (Adam). The patch is applied to a fixed corner of the image.

### Why this works

Neural network vision features are **non-robust**: small, structured changes to pixel values
can dramatically shift the activation patterns in the vision encoder without changing the
semantic content visible to humans. The patch exploits this sensitivity.


In [ ]:
# ── Create a sample "source" image to apply the patch to ────────────────────
# We use a solid colour image for clarity; the attack works on any image.

IMAGE_SIZE = 224  # CLIP-ViT-B/32 input size
PATCH_SIZE = 64   # patch covers a 64x64 corner (8% of the image area)

# Source: a plain blue-grey image (represents "neutral" content)
src_pil = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), color=(80, 100, 140))

# Draw a simple diagonal grid to make it look like a technical diagram
draw = ImageDraw.Draw(src_pil)
for i in range(0, IMAGE_SIZE, 20):
    draw.line([(0, i), (IMAGE_SIZE, i)], fill=(60, 80, 120), width=1)
    draw.line([(i, 0), (i, IMAGE_SIZE)], fill=(60, 80, 120), width=1)

# Define the target text: what we want CLIP to "see" after the patch is applied
TARGET_TEXT = 'a harmless technical diagram with calibration markers'
SOURCE_TEXT = 'a violent dangerous image'  # what we'd rather move away from

# Get target embedding (fixed during optimisation)
target_embed = get_text_embedding([TARGET_TEXT])  # (1, D)
source_embed = get_text_embedding([SOURCE_TEXT])  # (1, D)

# Compute baseline similarity (no patch)
orig_embed = get_image_embedding(src_pil)
sim_to_target_before = (orig_embed @ target_embed.T).item()
sim_to_source_before = (orig_embed @ source_embed.T).item()

print(f'Before patch:')
print(f'  Sim to "{TARGET_TEXT[:50]}...": {sim_to_target_before:.4f}')
print(f'  Sim to "{SOURCE_TEXT[:50]}":    {sim_to_source_before:.4f}')

In [ ]:
# ── CLIP preprocessing ───────────────────────────────────────────────────────
# CLIP normalises images to mean=(0.48145466, 0.4578275, 0.40821073)
#                            std =(0.26862954, 0.26130258, 0.27577711)

CLIP_MEAN = torch.tensor([0.48145466, 0.4578275,  0.40821073]).view(3,1,1).to(DEVICE)
CLIP_STD  = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3,1,1).to(DEVICE)

to_tensor = T.ToTensor()  # PIL -> [0,1] CHW tensor

def clip_normalise(t):
    return (t - CLIP_MEAN) / CLIP_STD

def embed_image_tensor(img_t):
    """img_t: [0,1] CHW tensor. Returns normalised CLIP embedding."""
    norm = clip_normalise(img_t.unsqueeze(0))  # add batch dim
    feats = model.vision_model(pixel_values=norm)[1]  # pooled output
    feats = model.visual_projection(feats)
    return torch.nn.functional.normalize(feats, dim=-1)


# ── Patch optimisation ───────────────────────────────────────────────────────

STEPS    = 200    # gradient ascent steps (increase for better attack, CPU: ~1–2 min)
LR       = 0.05   # Adam learning rate

# Convert source image to tensor
src_t = to_tensor(src_pil).to(DEVICE)  # (3, H, W)

# Patch starts as random noise in [0, 1]
patch = torch.rand(3, PATCH_SIZE, PATCH_SIZE, device=DEVICE, requires_grad=True)
optimizer = torch.optim.Adam([patch], lr=LR)

# Build patch mask: top-left corner
mask = torch.zeros_like(src_t)
mask[:, :PATCH_SIZE, :PATCH_SIZE] = 1.0

history_sim = []

for step in range(STEPS):
    optimizer.zero_grad()

    # Apply patch to source image
    patched = src_t * (1 - mask) + patch.clamp(0, 1) * mask

    # Get image embedding
    img_embed = embed_image_tensor(patched)

    # Maximise similarity to target text
    loss = -(img_embed @ target_embed.T).squeeze()

    loss.backward()
    optimizer.step()

    # Project patch values back to [0, 1]
    with torch.no_grad():
        patch.clamp_(0, 1)

    if step % 20 == 0 or step == STEPS - 1:
        sim_val = -loss.item()
        history_sim.append((step, sim_val))
        print(f'  Step {step:>4d} | Sim to target: {sim_val:.4f}')

print('\nOptimisation complete.')

In [ ]:
# ── Evaluate the attack ──────────────────────────────────────────────────────

with torch.no_grad():
    patched_t   = src_t * (1 - mask) + patch.clamp(0, 1) * mask
    final_embed = embed_image_tensor(patched_t)

sim_to_target_after = (final_embed @ target_embed.T).item()
sim_to_source_after = (final_embed @ source_embed.T).item()

print(f'After patch:')
print(f'  Sim to target text: {sim_to_target_after:.4f}  (was {sim_to_target_before:.4f}, +{sim_to_target_after - sim_to_target_before:.4f})')
print(f'  Sim to source text: {sim_to_source_after:.4f}  (was {sim_to_source_before:.4f}, {sim_to_source_after - sim_to_source_before:+.4f})')

# Convert final patched image and patch to PIL
def tensor_to_pil(t):
    return Image.fromarray((t.detach().cpu().permute(1,2,0).numpy() * 255).clip(0,255).astype('uint8'))

patched_pil = tensor_to_pil(patched_t)
patch_pil   = tensor_to_pil(patch.clamp(0, 1))


# ── Visualise ────────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 4, figure=fig, width_ratios=[2, 1, 2, 2])

ax1 = fig.add_subplot(gs[0])
ax1.imshow(src_pil)
ax1.set_title('Original Image', fontsize=11)
ax1.axis('off')

ax2 = fig.add_subplot(gs[1])
ax2.imshow(patch_pil)
ax2.set_title(f'Adversarial\nPatch ({PATCH_SIZE}×{PATCH_SIZE})', fontsize=11)
ax2.axis('off')

ax3 = fig.add_subplot(gs[2])
ax3.imshow(patched_pil)
ax3.set_title('Patched Image\n(top-left corner)', fontsize=11)
ax3.axis('off')

# Similarity bar chart
ax4 = fig.add_subplot(gs[3])
labels_bar = ['Target\n(benign)', 'Source\n(dangerous)']
before = [sim_to_target_before, sim_to_source_before]
after  = [sim_to_target_after,  sim_to_source_after]
x = np.arange(len(labels_bar))
w = 0.35
ax4.bar(x - w/2, before, w, label='Before patch', color='#5a7090', alpha=0.85)
ax4.bar(x + w/2, after,  w, label='After patch',  color='#00c8ff', alpha=0.85)
ax4.set_xticks(x)
ax4.set_xticklabels(labels_bar, fontsize=9)
ax4.set_ylabel('Cosine Similarity to CLIP text embedding')
ax4.set_title('Embedding shift', fontsize=11)
ax4.legend(fontsize=8)
ax4.set_ylim(0, 0.5)
ax4.axhline(0, color='white', lw=0.5)

plt.suptitle('CLIP Adversarial Patch: Shifting the Image Embedding Space', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Optimisation curve
steps_plt, sims_plt = zip(*history_sim)
plt.figure(figsize=(7, 3))
plt.plot(steps_plt, sims_plt, color='#00c8ff', linewidth=2)
plt.xlabel('Optimisation step')
plt.ylabel('Cosine similarity to target text')
plt.title('Patch optimisation curve — gradient ascent on CLIP similarity')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

---

## 4. Export the Adversarial Patch

Export the patch for use in Challenge 4 (Phantom Patch). Copy the output file to
`app/static/adversarial_patch.png` in the Docker environment.


In [ ]:
# ── Scale the patch to 200×200 for Challenge 4 download ─────────────────────

patch_export = patch_pil.resize((200, 200), Image.NEAREST)
patch_export.save('adversarial_patch.png')
print('Saved: adversarial_patch.png')
print()
print('To use in Challenge 4, copy to the Docker static directory:')
print('  cp adversarial_patch.png 08_llm_attacks_multimodal/app/static/adversarial_patch.png')
print()
print('Then rebuild the Docker image:')
print('  cd 08_llm_attacks_multimodal && docker compose up --build')

---

## 5. Why This Attack Matters for VLMs

LLaVA and similar VLMs use CLIP (or a CLIP-like encoder) as their visual front-end.
The image embeddings produced by the vision encoder are directly fed into the LLM.

**Attack surface:**
- A content safety classifier that runs on CLIP embeddings can be fooled by a patch
  that shifts the embedding toward benign-looking text anchors
- Patches are **transferable**: a patch trained on one CLIP variant often transfers to other models
- Patches are **universal**: a single patch can fool the classifier across many input images
  (this is the key result in arXiv:2502.07987)

**Why models are vulnerable:**
Neural network features are non-robust by construction — the geometry of high-dimensional
embedding spaces allows small pixel perturbations to make large jumps in embedding space.
This is a fundamental property of overparameterised networks trained with gradient descent.


---

## 6. Defences Against Adversarial Patches

### A. Feature Squeezing

Reduce the information density of the input before feeding it to the model.
High-frequency adversarial noise is destroyed by:
- Bit-depth reduction (quantise pixel values)
- Median or Gaussian filtering
- JPEG recompression


In [ ]:
# ── Feature squeezing demo ───────────────────────────────────────────────────
import io

def jpeg_squeeze(img_pil, quality=50):
    """Simulate JPEG recompression as a defence."""
    buf = io.BytesIO()
    img_pil.save(buf, 'JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).copy()

def bit_depth_reduce(img_pil, bits=4):
    """Reduce colour precision to 'bits' bits per channel."""
    factor = 256 // (2 ** bits)
    arr = np.array(img_pil)
    arr = (arr // factor) * factor
    return Image.fromarray(arr.astype('uint8'))

# Apply defences to the patched image
squeezed_jpeg = jpeg_squeeze(patched_pil, quality=40)
squeezed_bits = bit_depth_reduce(patched_pil, bits=4)

# Evaluate similarity after defence
squeezed_jpeg_embed = get_image_embedding(squeezed_jpeg)
squeezed_bits_embed = get_image_embedding(squeezed_bits)

sim_jpeg = (squeezed_jpeg_embed @ target_embed.T).item()
sim_bits = (squeezed_bits_embed @ target_embed.T).item()

print('Similarity to target after defence:')
print(f'  No defence:        {sim_to_target_after:.4f}')
print(f'  JPEG squeeze (q=40): {sim_jpeg:.4f}  ({sim_jpeg - sim_to_target_after:+.4f})')
print(f'  4-bit quantise:    {sim_bits:.4f}  ({sim_bits - sim_to_target_after:+.4f})')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (img, title) in zip(axes, [
    (patched_pil, f'Patched (sim={sim_to_target_after:.3f})'),
    (squeezed_jpeg, f'JPEG Q=40 (sim={sim_jpeg:.3f})'),
    (squeezed_bits, f'4-bit Quant (sim={sim_bits:.3f})'),
]):
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.suptitle('Feature Squeezing: defences vs. adversarial patch', fontsize=12)
plt.tight_layout()
plt.show()

### B. Adversarial Training

Adversarial training — the gold-standard empirical defence — augments the training data
with adversarial examples so the model learns robust features:

$$\min_\theta \; \mathbb{E}_{(x,y)} \left[ \max_{\delta \in \mathcal{S}} \ell(f_\theta(x + \delta), y) \right]$$

For VLMs this means fine-tuning the vision encoder (or its projection head) on adversarially
perturbed images while maintaining alignment with the original text descriptions.

**Trade-off:** Adversarial training consistently reduces clean accuracy. Models trained to be
robust to $L_\infty(\epsilon)$ perturbations typically lose 10–20% clean accuracy for
meaningful robustness gains (Madry et al., 2017).

### C. Randomised Smoothing

Randomised smoothing (Cohen et al., 2019) provides **certified** $L_2$ robustness:
- Add Gaussian noise $\mathcal{N}(0, \sigma^2 I)$ to the image at inference time
- The smoothed classifier's prediction at $x$ holds (with high probability) for all
  images within $L_2$ radius $r = \sigma \Phi^{-1}(p_A)$ of $x$
- Patches have large $L_2$ norms, so smoothing with $\sigma$ large enough to cover
  the patch provides certified defence

**Limitation for patches:** Patches occupy a bounded spatial region but have unbounded $L_\infty$
norm. The $L_2$ norm of a $64\times64$ patch at full intensity is
$\|\delta\|_2 \leq 64 \cdot \sqrt{3} \approx 110$, which requires $\sigma \approx 100$
to certify — destroying image quality. PatchSmooth (Salman et al., 2021) addresses
this by applying smoothing only over patch-shaped noise distributions.


In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────

print('=' * 65)
print('  ADVERSARIAL DEFENCE COMPARISON FOR VLMS')
print('=' * 65)
print(f'{"Defence":<25} {"Robustness":<15} {"Clean Acc":<15} {"Cost"}')
print('-' * 65)
rows = [
    ('Feature Squeezing',       'Heuristic',  'Slight drop',  'Zero (test-time)'),
    ('Adversarial Training',    'Empirical',  '10-20% drop',  'High (retrain)'),
    ('Randomised Smoothing',    'Certified L2','~15% drop',   'High (100× infer)'),
    ('PatchSmooth',             'Certified L2','~5% drop',    'Moderate'),
    ('Input Filtering (CLIP)',  'Heuristic',  'Minimal',      'Low (add step)'),
]
for name, robust, clean, cost in rows:
    print(f'{name:<25} {robust:<15} {clean:<15} {cost}')
print('=' * 65)

---

## 7. Key References

1. **Radford et al. (2021)**. *Learning Transferable Visual Models From Natural Language Supervision* (CLIP). [arXiv:2103.00020](https://arxiv.org/abs/2103.00020)

2. **Goodfellow et al. (2014)**. *Explaining and Harnessing Adversarial Examples* (FGSM). [arXiv:1412.6572](https://arxiv.org/abs/1412.6572)

3. **Madry et al. (2017)**. *Towards Deep Learning Models Resistant to Adversarial Attacks* (PGD). [arXiv:1706.06083](https://arxiv.org/abs/1706.06083)

4. **Cohen et al. (2019)**. *Certified Adversarial Robustness via Randomized Smoothing*. [arXiv:1902.02918](https://arxiv.org/abs/1902.02918)

5. **Qi et al. (2024)**. *Visual Adversarial Examples Jailbreak Aligned Large Language Models*. [arXiv:2306.13213](https://arxiv.org/abs/2306.13213)

6. **Gong et al. (2025)**. *FigStep: Jailbreaking Large Vision-Language Models via Scalable Typography-based Visual Prompts*. [arXiv:2311.05608](https://arxiv.org/abs/2311.05608)

7. **Rahmatullaev et al. (2025)**. *Universal Adversarial Attack on Aligned Multimodal LLMs*. [arXiv:2502.07987](https://arxiv.org/abs/2502.07987)
